# Experiment 1: Baseline

Compare retrieval and generated answers on the same 31 seeded eligible questions and fixed corpus, with separate diagnostic examples.

**Live execution is the default.** `MODE="live"` runs this notebook’s model calls with the configured OpenAI-compatible client. Set `OPENROUTER_API_KEY` before starting the kernel. Prepared document embeddings and frozen experimental inputs remain fixed; passage embeddings are not rebuilt.

**Study scope:** the frozen larger corpus contains **152,576 passages** associated with **15,585 fully covered question IDs** from MS MARCO v2.1 validation; this is a subset, not the full validation split. Experiment 1 uses a separate **10,210-passage corpus**, 31 seeded baseline questions, the effervescent diagnostic, and a separate IMEI diagnostic. Experiment 2 samples **500 opportunities + 200 controls**. Experiment 3 contains **79 sampled pairs, including IMEI**. Experiment 4 times **8 question pairs / 16 answers**. IMEI is excluded from the 31-question baseline averages but is already included in the frozen Experiment 2 and Experiment 3 samples; displaying it again adds no observation.

**Workflow:** **this notebook Experiment 1 (Baseline)** --> Diagnosis --> Experiment 2 (Targeted Reranking) --> Experiment 3 (Answer Quality) --> Experiment 4 (Latency Tradeoff)


## 1. Experiment setup

The table shows the main models and passage budgets. Expand the configuration below it for generation settings, endpoints, retry policies and the system prompt.

`configs/tutorial_inputs.json` declares the frozen inputs; `configs/experiment1.json` defines the scoring settings. The live setup displays the pipeline configuration before model calls.


In [1]:
STUDY = 'baseline'
from pathlib import Path
import os, sys
from IPython.display import display, Markdown, HTML
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "configs/tutorial_inputs.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.live_tutorial import TutorialRun
from shared_scoring.provider import embedding_payload, rerank_payload
from shared_scoring.tutorial_views import TutorialViews
from shared_scoring.notebook_display import display_table, details_html
from shared_scoring.io import read_json
import inspect, json
import pandas as pd

MODE = os.environ.get("NOTEBOOK_MODE", "live").lower()  # live | replay | plan
RUN_ID = os.environ.get("TUTORIAL_RUN_ID") or None
RESUME = os.environ.get("TUTORIAL_RESUME") == "1"  # explicit same-run resume only

tutorial = TutorialRun(PROJECT, STUDY, mode=MODE, run_id=RUN_ID, resume=RESUME)
tutorial.preflight(create_client=False)  # assets, credentials and offline tests; no model requests
client = None
if MODE == "live":
    from openai import OpenAI
    import httpx
    # Standard OpenAI-style client; the transport records this run and enforces caps.
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=tutorial.retrieval["api_base"],
        max_retries=0,
        timeout=tutorial.retrieval["timeout_seconds"],
        http_client=httpx.Client(transport=tutorial.transport()),
    )
    tutorial.bind_client(client)
# Selecting Live authorizes this notebook's declared workflow; no second paid-call switch.

view = TutorialViews(tutorial)
view.show("configuration")


,Setting,Value
0,Dataset,microsoft/ms_marco · v2.1 · validation
1,Embedding model,qwen/qwen3-embedding-4b
2,Reranker model,qwen/qwen3-reranker-8b
3,Generator model,qwen/qwen3-30b-a3b-instruct-2507
4,Candidate / context budget,30 / 5 passages
5,Passage text limit,"1,800 characters per passage"
6,Generation output limit,400 tokens


## 2. Verify the 31 seeded eligible questions

The selected question list, candidates, results and frozen IDs must agree. Inspect the questions below; the code verifies their order and membership.

Question counts and completion counts are checked separately; the baseline averages use only the 31 seeded questions.


In [2]:
if MODE == "plan":
    display(Markdown("Plan: verify the 31 seeded eligible questions and separate IMEI diagnostic."))
else:
    cohort = view.questions()
    baseline = cohort[~cohort.diagnostic]
    assert len(baseline) == 31 and baseline.query_id.is_unique
    assert cohort.loc[cohort.diagnostic, "query_id"].tolist() == ["1053665"]
    display_table(baseline[["query_id", "question"]].reset_index(drop=True))
    display(Markdown("**Verified: 31 eligible baseline questions.** IMEI (1053665) is checked separately and excluded from baseline averages."))


query_id,question
1088186,"weather in texarkana, arkansas"
query_id,question
1038185,what is the ideal credit card balance for credit score
1086238,what causes antibiotic resistance in bacteria
1100277,gestation medical definition
1029544,what is tracers
1082398,what does warbled mean
1055197,what is flex fuel on a jeep?
1089273,types of road hugger tires
1085733,what could cause soreness of heel


**Verified: 31 eligible baseline questions.** IMEI (1053665) is checked separately and excluded from baseline averages.

## 3. Load the questions and fixed corpus

The baseline corpus contains **10,210 deduplicated passages** associated with the **31 seeded baseline questions** and **1,000 additional distractor questions**. Distractor questions were sampled with seed 42 from the same 15,585 eligible IDs, excluding the baseline questions and the original effervescent diagnostic. All associated selected and unselected passages were included; individual distractor passages were not sampled independently and need not be irrelevant to every baseline question.

The dataset revision is pinned. Passages are trimmed, deduplicated by case-insensitive text, assigned stable IDs, and limited to 1,800 characters. The reconstructed corpus must match the fixed corpus.

The IMEI diagnostic **1053665 (what is imei and where do i find it)** is evaluated separately against this unchanged corpus, which contains **0 of its 2 annotated supporting passages**. Its answers and scores remain separate from the 31-question averages. **1100870 (effervescent meaning)** is a different diagnostic and is not relabeled as IMEI.


In [3]:
view.show('corpus')


,dataset,revision,baseline_questions,diagnostic_questions,corpus_passages,clipped_passages,diagnostic_gold_in_corpus,diagnostic_gold_total
0,microsoft/ms_marco,a47ee7aae8d7d466ba15f9f0bfac3b3681087b3a,31,1,10210,0,0,1


,query_id,scope,question,reference_answers,gold_ids
0,1088186,baseline_31,"weather in texarkana, arkansas",[The Texarkana forecast for Aug 30 is 76 degrees and Patchy rain possible. There is 83 percentage chance of rain and 13 mph winds from the Northeast. The Te...,[p_10af63d165c7c485aedb]
,query_id,scope,question,reference_answers,gold_ids
1,1038185,baseline_31,what is the ideal credit card balance for credit score,"[If you have a credit card with a $1,000 credit limit and a $100 balance then you are 10% “utilized” on that card.]",[p_40ecf58fe3664fcffaa5]
2,1086238,baseline_31,what causes antibiotic resistance in bacteria,"[If you take an antibiotic when you actually have a viral infection, the antibiotic attacks bacteria in your body bacteria that are either beneficial or at ...",[p_32a99c36bfc8d9cebdca]
3,1100277,baseline_31,gestation medical definition,[Gestation is defined as the period of fetal development from conception until birth.],[p_ded14ec74054e77e2a93]
4,1029544,baseline_31,what is tracers,[Tracers is a 2015 American action film.],[p_8c691bd8eb091d4fefd6]
5,1082398,baseline_31,what does warbled mean,"[Warble means to sing or whistle with trills, quavers, or melodic embellishments.]",[p_e11e11ab1f92dca78f46]
6,1055197,baseline_31,what is flex fuel on a jeep?,[Jeeps display a silver Flex Fuel E85 tag. E85 ethanol flex-fuel vehicles from Mercedes-Benz and Nissan do not carry any special identification.],[p_9e32bd1a8c1c2459bc87]
7,1089273,baseline_31,types of road hugger tires,[Road Hugger Radial GT is a type of Road Hugger tire.],[p_b7ef8ac8a31403454616]
8,1085733,baseline_31,what could cause soreness of heel,"[The causes of heel pain are plantar fasciitis, achilles tendinitis, achilles tendon rupture, bone tumor, bursitis Haglund's deformity, heel spur, osteomyel...",[p_daed0067e95cbd258615]


## 4. Document embedding and cache

The baseline reuses existing cached Qwen document embeddings for its **10,210 passages**. The matrix, origin record, and configuration document that reuse; live execution embeds questions without rebuilding document embeddings.

Document vectors are prepared inputs. The query-embedding interface below is used for fresh question embeddings in section 6; document embeddings are not rebuilt.


In [4]:
def embed_api(questions):
    # Actual OpenAI-style embedding call. Passage vectors remain prepared inputs.
    return client.embeddings.create(**embedding_payload(
        tutorial.retrieval["embedding_model"], questions, tutorial.retrieval["dimensions"],
        instruction=tutorial.retrieval["query_instruction"]))

view.show('document_embeddings')


{'cache_id': '95638881ae385e4871a34626ee33af57b4b556a95ed6c019b7958030f08263ea', 'origin': 'Fresh baseline document embeddings', 'creator_run_id': 'baseline-fresh-5413aa92e9c1375fa06f'}

Saved document embedding matrix: (10210, 2560)


In [5]:
display(Markdown("```python\n" + inspect.getsource(embedding_payload) + "\n```"))


### Cosine retrieval and cache verification

Haystack’s in-memory document store uses the prepared document vectors. Cosine similarity ranks thirty candidates; each RAG path supplies five passages to the generator. Input integrity is checked internally.

In [6]:
# Reuse the prepared document vectors; query embedding happens in section 6.
def retrieve_top30(query_embedding):
    retriever = tutorial.document_retriever()
    return retriever.run(query_embedding=list(query_embedding))["documents"]


## 5. Scoring implementation and definitions

| Metric | definition |
|---|---|
| **Recall@30 / Recall@5** | Unique known support IDs retrieved / all known support IDs |
| Hit@30 / Hit@5 | Any known support ID retrieved |
| MRR@5 | Reciprocal rank of first known support, else 0 |
| nDCG@5 | Binary DCG divided by ideal DCG for min(5, number of gold IDs) |
| Normalized EM | Any reference matches after normalization |
| **Token F1** | Maximum token multiset overlap F1 over references |
| Citation diagnostics | Known gold cited; count of cited IDs absent from selected five |
| Human rubric pass rate | Mean `correct` on questions with both successful arms and both reviews |

Normalization lowercases, removes single/grouped `[p_ID]` citations, removes punctuation and articles (`a`, `an`, `the`), then collapses whitespace. EM and F1 measure lexical agreement, not semantic correctness. Gold labels may be incomplete. Unreviewed human scores remain blank.

In [7]:
from shared_scoring.answers import score_custom_answer, normalize_custom_answer
display(Markdown("```python\n" + inspect.getsource(normalize_custom_answer) + "\n```"))
display(Markdown("```python\n" + inspect.getsource(score_custom_answer) + "\n```"))


```python
def normalize_custom_answer(answer, citation_pattern):
    """Reproduce the preserved custom normalization for cross-checking."""
    text = remove_passage_citations(answer.lower(), citation_pattern)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(re.sub(r'\b(a|an|the)\b', ' ', text).split())

```

```python
def score_custom_answer(answer, reference_answers, citation_pattern):
    """Return the existing custom exact match and Token F1 on the 0–1 scale."""
    if not reference_answers:
        raise ValueError('Supply all reference answers; an empty reference list is invalid.')
    normalized = normalize_custom_answer(answer, citation_pattern)
    tokens = normalized.split()
    f1 = []
    for reference in reference_answers:
        other = normalize_custom_answer(reference, citation_pattern).split()
        overlap = sum((Counter(tokens) & Counter(other)).values())
        f1.append(2 * overlap / (len(tokens) + len(other)) if tokens or other else 1.0)
    return {'normalized_em': float(any(normalized == normalize_custom_answer(r, citation_pattern) for r in reference_answers)),
            'token_f1': max(f1)}

```

## 6. Retrieval, evidence selection and generation

Follow the question through query embedding, Haystack retrieval of thirty candidates, selection of five passages with or without reranking, and two separate generator requests.

The cells below show the actual Qwen/OpenRouter calls through an OpenAI-compatible client. Live execution performs query embedding, retrieval, reranking and generation for this experiment’s questions.


In [8]:
# Fresh query embeddings and retrieval for 31 questions plus the separate IMEI diagnostic.
retrieval_run = tutorial.retrieve(embed_api, retrieve_api=retrieve_top30)
if retrieval_run is not None:
    display(retrieval_run)
    # The embedding path selects the first five of these thirty candidates.
    view.show("contexts")


Execution mode: execute | saved complete answers: 64


In [ ]:
def rerank_api(question, candidates):
    # OpenRouter rerank is exposed through the OpenAI client's generic POST API.
    return client.post("/rerank", cast_to=dict[str, object], body=rerank_payload(
        tutorial.retrieval["reranker_model"], question, candidates))

reranking_run = tutorial.rerank(rerank_api)
if reranking_run is not None:
    display(reranking_run)
retrieval_metrics_run = tutorial.diagnose()
if retrieval_metrics_run is not None:
    display(retrieval_metrics_run)


In [ ]:
def generate_api(messages, settings, routing=None):
    # The actual SDK call; the tutorial invokes it for each declared question/arm.
    return client.chat.completions.create(
        model=settings["model"], messages=messages,
        temperature=settings["temperature"], max_tokens=settings["max_tokens"],
        **settings["extras"], **({"extra_body": {"provider": routing}} if routing else {}))

generation_run = tutorial.generate(generate_api)
if generation_run is not None:
    display(generation_run)


## 7. Completed requests and failures

A complete pair means two successful answers to the same question, one per RAG path. The baseline contains 31 questions. The IMEI diagnostic is displayed separately later and is excluded from the 31-question averages.

Attempted questions, successful answers, complete pairs and failed requests have different denominators. Failed answers remain unavailable; they do not become zero scores.


In [9]:
view.show('completion')


,scope,planned_questions,attempted_questions,successful_pairs,questions_without_successful_pair,successful_baseline_answers,successful_reranked_answers,failed_or_blocked_arms
0,baseline_31,31,31,31,0,31,31,0
1,diagnostic,1,1,1,0,1,1,0


,query_id,scope,retrieval_status,baseline_status,reranked_status,successful_pair,baseline_generation_attempted,reranked_generation_attempted
0,1088186,baseline_31,ok,ok,ok,True,True,True
,query_id,scope,retrieval_status,baseline_status,reranked_status,successful_pair,baseline_generation_attempted,reranked_generation_attempted
1,1038185,baseline_31,ok,ok,ok,True,True,True
2,1086238,baseline_31,ok,ok,ok,True,True,True
3,1100277,baseline_31,ok,ok,ok,True,True,True
4,1029544,baseline_31,ok,ok,ok,True,True,True
5,1082398,baseline_31,ok,ok,ok,True,True,True
6,1055197,baseline_31,ok,ok,ok,True,True,True
7,1089273,baseline_31,ok,ok,ok,True,True,True
8,1085733,baseline_31,ok,ok,ok,True,True,True


No failed or blocked arms in this recorded batch.

,stage,status,requests
0,generation,ok,64
1,query_embedding,ok,32
2,rerank_top30,ok,32


## 8. Per-question scores and answers

Inspect retrieval coverage and answer overlap for each question. The scoring functions are defined in section 5. Questions that fail before retrieval remain explicit failed records with unavailable metrics.

Baseline averages use complete question pairs and exclude the diagnostic.


In [10]:
view.show('retrieval')
view.show('overlap')


,query_id,scope,arm,status,recall_at_30,hit_at_30,recall_at_5,hit_at_5,mrr_at_5,ndcg_at_5,normalized_em,token_f1
0,1088186,baseline_31,baseline,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.6667
,query_id,scope,arm,status,recall_at_30,hit_at_30,recall_at_5,hit_at_5,mrr_at_5,ndcg_at_5,normalized_em,token_f1
1,1088186,baseline_31,reranked,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.5366
2,1038185,baseline_31,reranked,ok,1.0,1.0,0.0,0.0,0.0000,0.0000,0.0,0.2333
3,1038185,baseline_31,baseline,ok,1.0,1.0,0.0,0.0,0.0000,0.0000,0.0,0.2250
4,1086238,baseline_31,baseline,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.3091
5,1086238,baseline_31,reranked,ok,1.0,1.0,1.0,1.0,0.5000,0.6309,0.0,0.2975
6,1100277,baseline_31,reranked,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.4681
7,1100277,baseline_31,baseline,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.4783
8,1029544,baseline_31,baseline,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.0769


Query ID,Arm,Answer / failure
1088186,baseline,"The weather in Texarkana, Arkansas on August 30 is expected to be 76 degrees with a chance of patchy rain (83% chance) and winds from the northeast at 13 mph. On August 31, the forecast is 75 degrees with light drizzle (96% chance of rain) and north winds at 9 mph [p_10af63d165c7c485aedb]."
1088186,reranked,"The current weather in Texarkana, Arkansas, as of August 30, is 76 degrees with a chance of patchy rain and 13 mph northeast winds. On August 31, the forecast is 75 degrees with light drizzle, a 96% chance of rain, and 9 mph north winds [p_10af63d165c7c485aedb]. The climate in Texarkana features hot summers, with July being the warmest month, and cold winters, with December being the coolest. The highest recorded temperature was 114 °F in 2007, and the lowest was 17 °F in 1990 [p_6f23571c07e9942f3b49]."
1038185,reranked,"The ideal credit card balance for a good credit score is one that keeps the credit utilization ratio at 20% or less of the available credit limit [p_cfc2436ca7fad93fec27]. For example, with a $10,000 credit limit, the balance should ideally not exceed $2,000 to maintain a favorable utilization rate [p_85a475075ec630d9edef]."
1038185,baseline,"The ideal credit card balance for a good credit score is one that keeps credit utilization at 20% or less of the credit limit [p_cfc2436ca7fad93fec27]. For example, with a $10,000 credit limit, carrying a balance of $3,000 or less is recommended for optimal credit scoring [p_85a475075ec630d9edef]. However, if the card does not report balances to credit bureaus (such as some true charge cards), utilization may not affect the score [p_b5fdfb1fff8942366ea0][p_2be27f8940733a0a7973]."
1086238,baseline,"Antibiotic resistance in bacteria is caused by the overuse and misuse of antibiotics, which promotes the development and spread of resistant properties. This can occur when antibiotics are used unnecessarily, such as for viral infections, leading to selective pressure that favors resistant bacteria. Additionally, bacteria can transfer resistance traits to other bacteria, accelerating the spread of resistance [p_595215c087c42dc57711][p_32a99c36bfc8d9cebdca][p_7e567c198906201dcfe1]."
1086238,reranked,"Antibiotic resistance in bacteria is primarily caused by the overuse and misuse of antibiotics, including taking them for viral infections when they are ineffective. This improper use allows harmless bacteria to develop resistance, which can be transferred to other bacteria, and creates opportunities for harmful bacteria to thrive. The repeated exposure to antibiotics also leads to the selection of resistant strains, such as those seen in staph infections [p_595215c087c42dc57711][p_32a99c36bfc8d9cebdca][p_519a18958ac4ed1fddd9][p_7e567c198906201dcfe1]."
1100277,reranked,"Gestation is the period of fetal development from conception until birth, also known as pregnancy, which lasts about nine months in humans [p_ded14ec74054e77e2a93][p_7e288e8d2ab7fccf7cbd]. It refers to the time a person or animal spends developing inside the mother's uterus before birth [p_74008bec93847cda08fb]."
1100277,baseline,"Gestation is the period of fetal development from conception until birth, also known as pregnancy, lasting approximately nine months in humans [p_ded14ec74054e77e2a93][p_7e288e8d2ab7fccf7cbd]. It refers to the time a person or animal spends developing inside the mother's uterus before birth [p_74008bec93847cda08fb]."
1029544,baseline,"Tracers can refer to multiple things based on the provided passages: 1. **A 2015 American action film** titled *Tracers*, directed by Daniel Benmayor and starring Taylor Lautner, Marie Avgeropoulos, Adam Rayner, and Rafi Gavron. It was released on March 20, 2015, and received mixed reviews, with a 27% rating on Rotten Tomatoes and a 45/100 on Metacritic [p_8c691bd8eb091d4fefd6][p_93d0cb28dd1a21b1eb61]. 2. **A character in the video game Overwatch** named Tracer (Lena Oxton), a former Overwatch agent known for 

## 9. Baseline results: 31 questions

Each automatic comparison uses the same complete baseline pairs. Completion counts and failures are reported separately. Retrieval coverage and answer overlap measure different things.

The new baseline is sampled from eligible labeled questions with seed 42. Support comes from dataset annotations; human review remains pending. The review rubric checks correctness and completeness, support from the supplied evidence, unsupported claims and appropriate uncertainty when evidence is insufficient.

Timing fields are retained here for inspection. **Experiment 4: Latency Tradeoff** measures response time separately.


In [11]:
view.show('retrieval_summary')
view.show('overlap_summary')


arm,Embedding RAG,Embedding + reranking RAG,Delta (reranked − baseline)
paired_questions,31.0,31.0,0.0
arm,Embedding RAG,Embedding + reranking RAG,Delta (reranked − baseline)
failed_questions,0.0,0.0,0.0
recall_at_30,1.0,1.0,0.0
hit_at_30,1.0,1.0,0.0
recall_at_5,0.9032,0.9032,0.0
hit_at_5,0.9032,0.9032,0.0
mrr_at_5,0.7231,0.7645,0.0414
ndcg_at_5,0.7645,0.8001,0.0356
normalized_em,0.0,0.0,0.0


Denominator: 31 complete pairs out of 31 attempted baseline questions.
Human review CSV: .cache/qwen-runs/baseline/5413aa92e9c1375fa06f/runs/baseline-fresh-5413aa92e9c1375fa06f/manual_review.csv


,recall_at_30_delta,hit_at_30_delta,recall_at_5_delta,hit_at_5_delta,mrr_at_5_delta,ndcg_at_5_delta,normalized_em_delta,token_f1_delta
query_id,,,,,,,,
1029167,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,0.0301
,recall_at_30_delta,hit_at_30_delta,recall_at_5_delta,hit_at_5_delta,mrr_at_5_delta,ndcg_at_5_delta,normalized_em_delta,token_f1_delta
query_id,,,,,,,,
1029544,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,-0.0488
1032146,0.0,0.0,0.0,0.0,0.0000,0.1228,0.0,0.0049
1033042,0.0,0.0,0.0,0.0,0.1667,0.1309,0.0,0.0000
1038185,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,0.0083
1040180,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,0.0000
1050417,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,-0.0494


### Standard answer evaluation

Evaluation attribution: Hugging Face Evaluate’s SQuAD metric and RAGAS Faithfulness and Answer relevancy.

SQuAD normalized exact match and **Token F1** use **0–100 percentages**; the original answer scorer above uses **0–1**. **Annotated-support Recall@5** remains a separate retrieval metric on **0–1**. **Faithfulness** uses **0–1**; **Answer relevancy** is usually 0–1, with a mathematical cosine range of −1–1. RAGAS judges evidence support and question relevance; it is not human correctness.

Scores below use the same complete baseline pairs and each arm’s exact separate five passages. Truncated answers remain included. Generation failures and earlier judge failures remain inspectable. Diagnostic scores remain separate. Live execution computes the standard scores after generation.


In [12]:
# Score only answers produced by this RUN_ID. Live includes RAGAS where declared.
score_run = await tutorial.score()
if score_run is not None:
    display(score_run)


1088186 baseline answer_relevancy: ok 0.7879529392011603 (11.0s)


1088186 baseline faithfulness: ok 1.0 (19.2s)


1088186 reranked answer_relevancy: ok 0.7995303559974372 (13.2s)


1088186 reranked faithfulness: ok 1.0 (26.7s)


1038185 reranked faithfulness: ok 0.5 (11.2s)


1038185 reranked answer_relevancy: ok 0.9448827208128514 (12.5s)


... (115 outputs left out)

1084137 baseline answer_relevancy: ok 0.9419695261841895 (11.6s)


1084137 reranked answer_relevancy: ok 0.9022043952859288 (13.5s)


1084137 reranked faithfulness: ok 1.0 (30.4s)


{'saved_run': 'baseline-fresh-5413aa92e9c1375fa06f',
 'baseline_questions': 31,
 'complete_baseline_pairs': 31,
 'saved_judge_outcomes': 124}

In [13]:
from shared_scoring.answers import score_squad_answers
from shared_scoring.judge import run_ragas_evaluation
display(Markdown("```python\n" + inspect.getsource(score_squad_answers) + "\n```"))
display(Markdown("```python\n" + inspect.getsource(run_ragas_evaluation) + "\n```"))


def load_squad_metric(metric_source='squad', cache_directory=None):
    """Load Hugging Face SQuAD explicitly; this is never called by replay."""
    if cache_directory:
        directory = Path(cache_directory).resolve()
        os.environ.setdefault('HF_HOME', str(directory / 'huggingface'))
        os.environ.setdefault('HF_MODULES_CACHE', str(directory / 'modules'))
        os.environ.setdefault('HF_EVALUATE_CACHE', str(directory / 'evaluate'))
    import evaluate
    return evaluate.load(str(metric_source), cache_dir=str(cache_directory) if cache_directory else None)

def create_ragas_metrics(client, embeddings, settings):
    """Construct the real RAGAS metrics only for explicit judge evaluation."""
    from ragas.llms import llm_factory
    from ragas.metrics.collections import Faithfulness, AnswerRelevancy
    llm = llm_factory(settings['judge_model'], provider='openai', adapter='instructor', client=client,
        temperature=settings['judge_temperature'], top_p=settings['judge_top_p'], max_tokens=settings['judge_max_tokens'],
        seed=settings['judge_seed'], max_retries=settings.get('structured_output_max_attempts', 1),
        system_prompt=settings['judge_system_prompt'])
    available = {'faithfulness': Faithfulness(llm=llm),
                 'answer_relevancy': AnswerRelevancy(llm=llm, embeddings=embeddings, strictness=settings['relevancy_strictness'])}
    unknown = set(settings['metrics']) - available.keys()
    if unknown:
        raise ValueError(f'Unsupported configured RAGAS metrics: {sorted(unknown)}')
    return {name: available[name] for name in settings['metrics']}

In [14]:
view.show('scores')
view.show('judge_outcomes')


,Metric,Scale,Successful matching pairs,Embedding RAG,Embedding + reranking RAG,Paired delta (reranking − embedding),Improved,Tied,Lower,Delta bootstrap 95% low,Delta bootstrap 95% high
0,Annotated-support Recall@5,0–1; existing retrieval metric,31,0.9032,0.9032,0.0000,0,31,0,0.0000,0.0000
1,SQuAD normalized exact match,0–100 percent,31,0.0000,0.0000,0.0000,0,31,0,0.0000,0.0000
2,Token F1,0–100 percent,31,37.8523,35.9012,-1.9512,7,8,16,-4.1769,0.0557
3,Faithfulness,0–1,31,0.9485,0.9275,-0.0210,4,22,5,-0.0807,0.0379
4,Answer relevancy,usually 0–1; cosine can be −1–1,31,0.7997,0.7885,-0.0112,7,17,7,-0.0316,0.0032


,query_id,path,squad_exact_match_pct,squad_token_f1_pct,faithfulness_0_1,answer_relevancy_0_1,truncated
0,1088186,Embedding RAG,0.0,66.666667,1.000000,0.787953,False
,query_id,path,squad_exact_match_pct,squad_token_f1_pct,faithfulness_0_1,answer_relevancy_0_1,truncated
1,1088186,Embedding + reranking RAG,0.0,53.658537,1.000000,0.799530,False
2,1038185,Embedding + reranking RAG,0.0,23.333333,0.500000,0.944883,False
3,1038185,Embedding RAG,0.0,22.500000,1.000000,0.883825,False
4,1086238,Embedding RAG,0.0,30.909091,1.000000,0.983508,False
5,1086238,Embedding + reranking RAG,0.0,29.752066,1.000000,0.912520,False
6,1100277,Embedding + reranking RAG,0.0,46.808511,1.000000,0.897838,False
7,1100277,Embedding RAG,0.0,47.826087,1.000000,0.897838,False
8,1029544,Embedding RAG,0.0,7.692308,1.000000,0.873739,False


,query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1


,case,SQuAD Token F1 (0–100),custom Token F1 (0–1)
0,empty answer and empty reference,0,1


SQuAD empty/empty Token F1 is 0; the original custom scorer returns 1.
All reference answers participate in SQuAD multiple-reference scoring.


### IMEI diagnostic scoring — separate from the 31-question baseline averages

This overview displays the same IMEI records inspected in detail in section 11. Compare both paths for **1053665, “what is imei and where do i find it”**, using the unchanged **10,210-passage corpus**. Neither annotated support is present. The diagnostic’s new answers and deterministic scores are reported separately; the original 31-question averages are unchanged.


In [1]:
view.show('walkthrough', contexts=False)


### IMEI diagnostic — 10,210-passage baseline corpus

**1053665: what is imei and where do i find it**

**0 of 2 annotated supporting passages are in this corpus.** Both retrieval paths therefore have annotated-support Recall@5 = 0. This separate diagnostic is excluded from the **31-question baseline averages**.

#### Embedding RAG

#### Embedding + reranking RAG

#### Embedding RAG

**Answer:**

#### Embedding + reranking RAG

**Answer:**

**Saved diagnostic scores:** SQuAD percentages. No new judge scores were requested for this small-corpus diagnostic.

,path,squad_token_f1_pct,squad_exact_match_pct,annotated_support_recall_at_5
0,Embedding RAG,8.333333,0.0,0.0
1,Embedding + reranking RAG,27.027027,0.0,0.0


## 10. Haystack evaluator cross-check

Compare the Haystack recall and MRR with the calculated retrieval scores. The evaluator uses passage IDs and multi-hit recall. Raw exact match is reported separately from normalized exact match.


In [17]:
view.show('haystack_crosscheck')


,haystack_recall_at_5,haystack_mrr_at_5,haystack_raw_em
arm,,,
baseline,0.9032,0.7231,0.0
reranked,0.9032,0.7645,0.0


## 11. Inspect the separate IMEI diagnostic

This detailed view revisits the same IMEI diagnostic shown in the scoring overview above. Trace **1053665** through the unchanged **10,210-passage baseline corpus**. It contains **0 of 2 annotated supports**. Compare each path’s passages, prompts, answers and diagnostic scores. No human correctness labels are assigned. Later notebooks show the larger-corpus recovery.


In [1]:
view.show('walkthrough')


{'query_id': '1053665',
 'question': 'what is imei and where do i find it',
 'corpus_passages': 10210,
 'annotated_supports_present': 0,
 'annotated_supports_total': 2,
 'excluded_from_baseline_averages': True}

### Query embedding → exact baseline top 30

Live execution embeds the question for local Haystack cosine retrieval over the fixed baseline document vectors. No passage embedding request is made. Inspect the thirty candidates below.

In [1]:
view.show('contexts')


{'query_embedding_origin': '.cache/qwen-runs/cached-subset/84556ae3e8e650a930be/requests/98ee3e8bf63673324b3b29350c34bd4f707fefc43e2c0ee1ddb4090e8bd2e6f8/attempt-01.json',
 'embedding_dimensions': 2560,
 'new_embedding_calls': 0}

,embedding_rank,passage_id,embedding_score,text
0,1,p_5a45997fb6bc4a13b57a,0.423135,Wireless phone number; Account holder billing ...
,embedding_rank,passage_id,embedding_score,text
1,2,p_dd5c85e0ab29e48c2997,0.392935,"To find your school's Site Code, please log in..."
2,3,p_998fb287cf11bc761fb7,0.391720,Simply Log On with your mobile phone number an...
3,4,p_19cdb207b1973a1a8aa7,0.387059,Create or access your user profile by entering...
4,5,p_82a8668aab88149e69cf,0.386614,"To find your school's Site Code, please log in..."
5,6,p_88166e9d68d3c5c8e8e9,0.380885,Enter your cell phone number and we will text ...
6,7,p_c4c226b47c00de128a55,0.375814,"Find a USB port. On laptops, they are typicall..."
7,8,p_d91ea8a15403bece17ba,0.372516,If you are using BlackBerry Device software ve...
8,9,p_77fc23c95b90963fcafa,0.368070,Please enter the customer number and access co...


### IMEI: embedding top five and reranked top five

These are the separate diagnostic’s contexts from the unchanged 10,210-passage corpus. Neither contains an annotated support passage.

In [1]:
# Inspect both selected fives from this run without repeating the reranker call.
if MODE != "plan":
    diagnostic_candidates = view.context_rows("1053665")
    for rank_column in ["rank", "rerank_rank"]:
        display_table(diagnostic_candidates.sort_values(rank_column).head(5))


### IMEI diagnostic — 10,210-passage baseline corpus

**1053665: what is imei and where do i find it**

**0 of 2 annotated supporting passages are in this corpus.** Both retrieval paths therefore have annotated-support Recall@5 = 0. This separate diagnostic is excluded from the **31-question baseline averages**.

#### Embedding RAG

#### Embedding + reranking RAG

### IMEI: exact contexts, prompts and two new diagnostic answers

The generator receives only each path’s own five passages. These diagnostic-only answers and scores remain separate from the 31-question averages.

In [1]:
view.show('prompts')
view.show('walkthrough')


### IMEI diagnostic — 10,210-passage baseline corpus

**1053665: what is imei and where do i find it**

**0 of 2 annotated supporting passages are in this corpus.** Both retrieval paths therefore have annotated-support Recall@5 = 0. This separate diagnostic is excluded from the **31-question baseline averages**.

#### Embedding RAG

**Answer:**

#### Embedding + reranking RAG

**Answer:**

**Saved diagnostic scores:** SQuAD percentages. No new judge scores were requested for this small-corpus diagnostic.

,path,squad_token_f1_pct,squad_exact_match_pct,annotated_support_recall_at_5
0,Embedding RAG,8.333333,0.0,0.0
1,Embedding + reranking RAG,27.027027,0.0,0.0


### IMEI: evaluation of the diagnostic answers

SQuAD measures reference overlap, not evidence support. Annotated-support Recall@5 is zero in both paths because both annotated supports are absent from the corpus. These scores do not enter the 31-question averages.

In [1]:
if MODE != "plan":
    diagnostic_scores = view.scores()
    display_table(diagnostic_scores[diagnostic_scores.query_id.eq("1053665")])


Separate IMEI diagnostic; original 31-question averages unchanged. This display makes no model calls.


,path,squad_token_f1_pct,squad_exact_match_pct,annotated_support_recall_at_5
0,Embedding RAG,8.333333,0.0,0.0
1,Embedding + reranking RAG,27.027027,0.0,0.0


## 12. Validate the run

Validate question IDs, contexts, request coverage, model settings, paired denominators and scoring provenance. Keep the 31 baseline questions and the separate IMEI diagnostic distinct, and verify the frozen input snapshot.

In [23]:
view.show('audit')
tutorial.show()


{'batch_question_count': 32,
 'baseline_attempted_questions': 31,
 'baseline_successful_pairs': 31,
 'human_review': 'pending',
 'judge_evaluation': 'explicit action; pending',
 'original_run_preserved': True}

Saved run validated; original artifacts were not rewritten.
